In [ ]:
!pip install rasterio
import rasterio
import numpy as np
import pandas as pd
import os 
from collections import defaultdict,Counter
from tqdm.auto import tqdm
import warnings
import json
import matplotlib.pyplot as plt
from math import ceil


warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

In [ ]:

def safe_serialize(obj):
    if isinstance(obj, (list, tuple)):
        return [safe_serialize(x) for x in obj]
    elif isinstance(obj, dict):
        return {k: safe_serialize(v) for k, v in obj.items()}
    elif hasattr(obj, '__dict__'):
        return str(obj)
    elif isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

def read_tif_metadata(filepath):
    try:
        with rasterio.open(filepath) as src:
            meta = {
                'Dimensiones': src.shape,
                'bandas': src.count,
                'CRS': str(src.crs) if src.crs else "unknown",
                'Resolución': [float(x) for x in src.res],
                'límites': [float(x) for x in src.bounds],
                'Driver': src.driver,
                'Compresión': str(src.compression) if src.compression else None,
                'Tipo de dato': str(src.dtypes[0]),
                'Nodata_value': safe_serialize(src.nodata),
                'Descripciones': list(src.descriptions) if src.descriptions else None,
            }

            tags = dict(src.tags()) if src.tags() else None
            profile = safe_serialize(dict(src.profile))

            return {
                'metadata': meta,
                'tags': tags,
                'profile': profile
            }
    except Exception as e:
        return {'error': str(e)}


carpetas = ['ESRI_LULC', 'FirePred', 'VIIRS_Day', 'VIIRS_Night']
base = '/kaggle/input/ts-satfire/ts-satfire/'

print("Analizando estructura de directorios...")
all_dirs = [os.path.join(base, d) for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))]
total_dirs = len(all_dirs)


data = {
    "dataset": "TS-SatFire",
    "base_path": base,
    "total_directories": total_dirs,
    "directories": {},
    "summary": {
        "global_stats": defaultdict(int),
        "dimensiones": defaultdict(int),
        "bandas": defaultdict(int),
        "crs": defaultdict(int),
    }
}


for dir_path in tqdm(all_dirs, desc="Directorios"):
    dir_name = os.path.basename(dir_path)
    data["directories"][dir_name] = {
        "path": dir_path,
        "folders": {}
    }

    for folder in carpetas:
        folder_path = os.path.join(dir_path, folder)
        exists = os.path.isdir(folder_path)
        data["directories"][dir_name]["folders"][folder] = {
            "exists": exists,
            "tif_count": 0,
            "files": []
        }

        if not exists:
            continue

        tif_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.tif', '.tiff'))]
        data["directories"][dir_name]["folders"][folder]["tif_count"] = len(tif_files)

        for tif_name in tqdm(tif_files, desc=f"{dir_name}/{folder}", leave=False):
            tif_path = os.path.join(folder_path, tif_name)
            info = read_tif_metadata(tif_path)
            
            file_entry = {
                "name": tif_name,
                "path": tif_path
            }
            file_entry.update(info)  

            data["directories"][dir_name]["folders"][folder]["files"].append(file_entry)


            if 'error' not in info:
                meta = info['metadata']
                dims = f"{meta['Dimensiones'][0]}x{meta['Dimensiones'][1]}"
                
                data["summary"]["global_stats"]["total_tif_files"] += 1
                data["summary"]["dimensiones"][dims] += 1
                data["summary"]["bandas"][meta['bandas']] += 1
                data["summary"]["crs"][meta['CRS']] += 1
                data["summary"]["global_stats"][f"tiene_{folder}"] = data["summary"]["global_stats"].get(f"tiene_{folder}", 0) + 1


print("\nGuardando resultados...")

with open('metadata_analysis.json', 'w') as f:
    json.dump(data, f, indent=2, default=str)


In [ ]:

data = json.load(open('/kaggle/working/metadata_analysis.json'))

In [ ]:

def analyze(data, union=set()):
    total = 0
    seq_day = []
    seq_night = []
    fire_pred = []
    path_valid = set()
    nis = set()  

    for dir_name, dir_info in data["directories"].items():
        path = dir_info["path"]
        
        if path in union:
            continue
            
        path_valid.add(path)

        folders = dir_info["folders"]

        lulc_info = folders.get("ESRI_LULC", {})
        has_lulc = lulc_info.get("exists", False) and lulc_info.get("tif_count", 0) > 0
        if not has_lulc:
            nis.add(path)

        day_count = folders.get("VIIRS_Day", {}).get("tif_count", 0)
        night_count = folders.get("VIIRS_Night", {}).get("tif_count", 0)
        pred_count = folders.get("FirePred", {}).get("tif_count", 0)

        seq_day.append(day_count)
        seq_night.append(night_count)
        fire_pred.append(pred_count)

        total += day_count + night_count + pred_count

    seq_day = np.array(seq_day)
    seq_night = np.array(seq_night)
    fire_pred = np.array(fire_pred)

    return seq_day, seq_night, fire_pred, nis, total, path_valid

seq_day,seq_night,fire_pred,nis,total,path_valid = analyze(data)

In [ ]:
len(nis)


In [ ]:

paths_list = list(path_valid) 


mask = (seq_day > 0) & (seq_night > 0) & (fire_pred > 0)

seq_day_clean   = seq_day[mask]
seq_night_clean = seq_night[mask]
fire_pred_clean = fire_pred[mask]
paths_clean     = [p for i, p in enumerate(paths_list) if mask[i]]


In [ ]:
fire_delete = set()

for dir_name, dir_info in data["directories"].items():
    folders = dir_info["folders"]
    
    tiene_fire = folders.get("FirePred", {}).get("exists", False)
    tiene_day  = folders.get("VIIRS_Day", {}).get("exists", False)
    
    if tiene_fire and not tiene_day:
        fire_delete.add(dir_info["path"]) 

In [ ]:
fire_delete1 = set()
for dir_name, dir_info in data["directories"].items():
    folders = dir_info["folders"]
    
    tiene_fire = folders.get("FirePred", {}).get("exists", False)
    tiene_day  = folders.get("VIIRS_Night", {}).get("exists", False)
    
    if tiene_fire and not tiene_day:
        fire_delete1.add(dir_info["path"]) 

In [ ]:
delete = set()

for dir_name, dir_info in data["directories"].items():
    folders = dir_info["folders"]
    
    has_day   = folders.get("VIIRS_Day", {}).get("exists", False)
    has_night = folders.get("VIIRS_Night", {}).get("exists", False)
    has_fire  = folders.get("FirePred", {}).get("exists", False)
    
    if not (has_day and has_night and has_fire):
        delete.add(dir_info["path"])  
        continue
    
    day_count   = folders["VIIRS_Day"]["tif_count"]
    night_count = folders["VIIRS_Night"]["tif_count"]
    
    if day_count != night_count:
        delete.add(dir_info["path"])

In [ ]:
union = nis.union(fire_delete.union(delete).union(fire_delete1))
len(union)

In [ ]:
seq_day,seq_night,fire_pred,_,total,path_valid = analyze(data,union=union)

In [ ]:
plt.hist(seq_day.T,bins=50)

In [ ]:
total

In [ ]:
todos_iguales = np.all([seq_day == seq_night, seq_night == fire_pred], axis=0)
all(todos_iguales)

In [ ]:
len(path_valid)

In [ ]:
def analyze_shapes(data, paths=None):
    """
    Extrae dimensiones, número de bandas y rutas de todos los archivos TIFF.
    
    Retorna (para cada categoría):
        shapes_X, bands_X, paths_X
    donde:
        shapes_X[i], bands_X[i], paths_X[i] → corresponden al mismo archivo.
    """
    # Inicializar listas: shapes, bands, paths
    shapes_day, bands_day, paths_day = [], [], []
    shapes_night, bands_night, paths_night = [], [], []
    shapes_pred, bands_pred, paths_pred = [], [], []
    shapes_lulc, bands_lulc, paths_lulc = [], [], []

    if paths is None:
        dir_items = data["directories"].items()
    else:
        dir_items = []
        for p in paths:
            if p in data["directories"]:
                dir_items.append((p, data["directories"][p]))
            else:
                for name, info in data["directories"].items():
                    if info["path"] == p:
                        dir_items.append((name, info))
                        break

    for dir_name, dir_info in dir_items:
        folders = dir_info["folders"]
        
        # Mapeo: folder → (shapes_list, bands_list, paths_list)
        mapping = {
            "VIIRS_Day":   (shapes_day,   bands_day,   paths_day),
            "VIIRS_Night": (shapes_night, bands_night, paths_night),
            "FirePred":    (shapes_pred,  bands_pred,  paths_pred),
            "ESRI_LULC":   (shapes_lulc,  bands_lulc,  paths_lulc)
        }
        
        for folder_name, (shape_list, band_list, path_list) in mapping.items():
            folder_data = folders.get(folder_name, {})
            if not folder_data.get("exists", False):
                continue
                
            for file_info in folder_data.get("files", []):
                meta = file_info.get("metadata", {})
                dim = meta.get("Dimensiones")
                bands = meta.get("bandas")
                filepath = file_info.get("path")
                
                if dim is not None and bands is not None and filepath is not None:
                    shape_list.append(tuple(dim))  # (alto, ancho)
                    band_list.append(int(bands))
                    path_list.append(filepath)
                else:
                    print(f"Datos incompletos en {folder_name} / {dir_name}")

    return (
        # VIIRS_Day
        shapes_day, bands_day, paths_day,
        # VIIRS_Night
        shapes_night, bands_night, paths_night,
        # FirePred
        shapes_pred, bands_pred, paths_pred,
        # ESRI_LULC
        shapes_lulc, bands_lulc, paths_lulc
    )

In [ ]:
(
    shapes_day, bands_day, paths_day,
    shapes_night, bands_night, paths_night,
    shapes_pred, bands_pred, paths_pred,
    shapes_lulc, bands_lulc, paths_lulc
) = analyze_shapes(data,paths=path_valid)

In [ ]:

def check_consistency(shapes, bands, name):
    if not shapes:
        print(f"{name}: sin datos")
        return
    
    shape_counts = Counter(shapes)
    band_counts = Counter(bands)
    
    print(f"\n{name}: {len(shapes)} archivos")
    print(f"  Dimensiones: {dict(shape_counts)}")
    print(f"  Bandas:      {dict(band_counts)}")
    

check_consistency(shapes_day, bands_day, "VIIRS_Day")
check_consistency(shapes_pred, bands_pred, "FirePred")
check_consistency(shapes_night, bands_night, "VIIRS_Night")
check_consistency(shapes_lulc, bands_lulc, "ESRI_LULC")

In [ ]:
problematic = [
    (dim, bands, path)
    for dim, bands, path in zip(shapes_night, bands_night, paths_night)
    if bands != 2
]
nproblematic = [
    (dim, bands, path)
    for dim, bands, path in zip(shapes_night, bands_night, paths_night)
    if bands == 2
]
if problematic:
    print(f"VIIRS_Night con bandas = 5: {len(problematic)}")
    for _, b, p in problematic:
        print(f"  {b} bandas → {p}")

In [ ]:

def plot_problematic_stats(problematic_list, num_bands=10, cols=5):

    all_stats = []
    file_names = []
    
    print(f"Procesando {len(problematic_list)} archivos...")
    
    for item in problematic_list:
      
        path = item[2] 
        file_names.append(os.path.basename(path))
        
        try:
            with rasterio.open(path) as src:
                stats = []
     
                limit = min(src.count, num_bands)
                for i in range(1, limit + 1):
                    band = src.read(i)

                    if src.nodata is not None:
                        valid = band[band != src.nodata]
                    else:
                        valid = band[~np.isnan(band)]
                    
                    stats.append(valid.mean() if valid.size > 0 else 0)
                
                while len(stats) < num_bands:
                    stats.append(0)
                all_stats.append(stats)
        except Exception as e:
            print(f"Error en {path}: {e}")
            all_stats.append([0] * num_bands)

    num_files = len(all_stats)
    rows = ceil(num_files / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
    axes = axes.flatten()

    for i in range(len(axes)):
        if i < num_files:

            bars = axes[i].bar(range(1, num_bands + 1), all_stats[i], color='skyblue', edgecolor='navy')
            axes[i].set_xticks(range(1, num_bands + 1))
            axes[i].set_xlabel("Bandas")
            axes[i].set_ylabel("Media")
            
            axes[i].tick_params(axis='both', which='major', labelsize=8)
        else:

            axes[i].axis('off')

    plt.tight_layout()
    plt.savefig("problematic_files_bands_report.png", dpi=150)
    plt.show()


plot_problematic_stats(problematic, num_bands=5, cols=10)

In [ ]:
def inspect_problematic_files(path):
    """
    Inspecciona un archivo raster y muestra toda su metadata y estadísticas.
    
    Args:
        path: Ruta al archivo raster
    """
    try:
        with rasterio.open(path) as src:
            print(f"\n  Dimensiones: {src.shape} (alto × ancho)")
            print(f"   Bandas: {src.count}")
            print(f"   CRS: {src.crs or 'No georreferenciado'}")
            print(f"   Resolución: {src.res} (grados o m/px)")
            print(f"   Tipo de dato: {src.dtypes[0]}")
            print(f"   Nodata: {src.nodata}")
            print(f"   Driver: {src.driver}")
            print(f"   Transform: {list(src.transform)}")
            print(f"   description: {src.descriptions}")
            
            for i in range(1, src.count + 1):
                try:
                    band = src.read(i)
                    
                    if src.nodata is not None:
                        valid = band[band != src.nodata]
                    else:
                        valid = band[~np.isnan(band)]
                    
                    if valid.size > 0:
                        print(f"      Banda {i}: min={valid.min():.2f}, max={valid.max():.2f}, "
                              f"mean={valid.mean():.2f}, std={valid.std():.2f}")
                    else:
                        print(f"      Banda {i}: sin datos válidos")
                        
                except Exception as e:
                    print(f"      Banda {i}: error al leer → {e}")
    
    except Exception as e:
        print(f"    Error al abrir: {e}")
    
    print()

In [ ]:
for _,_,path in problematic:

    inspect_problematic_files(path)

In [ ]:

def analyze_fire_loss_by_center_crop(dataset_instance, crop_size=256):
    """
    Escanea las imágenes originales antes del recorte para calcular cuántos 
    píxeles de fuego quedan fuera del área central.
    """
    total_fire_pixels_full = 0
    total_fire_pixels_crop = 0
    loss_per_sample = []
    

    print(f"Analizando pérdida de fuego (Crop: {crop_size}x{crop_size})...")
    
    for region_idx, folders in dataset_instance.raw_paths.items():
        tiffs = folders["VIIRS_Day"]
        
        for path in tqdm(tiffs, desc=f"Región {region_idx}"):
            with rasterio.open(path) as src:
                full_fire_mask = src.read(7)
                full_fire_mask = np.nan_to_num(full_fire_mask, nan=0.0)
                fire_count_full = np.sum(full_fire_mask > 0)
                
                y0 = (src.height - crop_size) // 2
                x0 = (src.width - crop_size) // 2
                win = rasterio.windows.Window(x0, y0, crop_size, crop_size)

                crop_fire_mask = src.read(7, window=win)
                crop_fire_mask = np.nan_to_num(crop_fire_mask, nan=0.0)
                fire_count_crop = np.sum(crop_fire_mask > 0)
                
                total_fire_pixels_full += fire_count_full
                total_fire_pixels_crop += fire_count_crop
                
                if fire_count_full > 0:
                    loss_pct = 100 * (1 - (fire_count_crop / fire_count_full))
                    loss_per_sample.append(loss_pct)

    pixels_lost = total_fire_pixels_full - total_fire_pixels_crop
    total_loss_pct = 100 * (pixels_lost / total_fire_pixels_full) if total_fire_pixels_full > 0 else 0
    
    print("\n" + "="*40)
    print(f"REPORTE DE PÉRDIDA POR CROP CENTRAL")
    print("-" * 40)
    print(f"Píxeles de fuego totales (Full): {total_fire_pixels_full}")
    print(f"Píxeles de fuego capturados (Crop): {total_fire_pixels_crop}")
    print(f"Píxeles perdidos: {pixels_lost}")
    print(f"PORCENTAJE DE PÉRDIDA TOTAL: {total_loss_pct:.2f}%")
    print("="*40)

    if loss_per_sample:
        plt.figure(figsize=(10, 5))
        plt.hist(loss_per_sample, bins=20, color='orangered', edgecolor='black')
        plt.title("Distribución de Pérdida de Fuego por Muestra")
        plt.xlabel("Porcentaje de Fuego Perdido (%)")
        plt.ylabel("Número de Imágenes")
        plt.grid(axis='y', alpha=0.3)
        plt.show()

    return total_loss_pct

loss = analyze_fire_loss_by_center_crop(train_dataset, crop_size=256)

In [ ]:
def analyze_dataset_balance(py_dataset, name="Dataset"):
    """
    Analiza la proporción de píxeles de fuego vs fondo en las etiquetas (Y)
    de un dataset ya procesado.
    """
    total_pixels = 0
    total_fire_pixels = 0
    samples_with_fire = 0
    
    print(f"--- Analizando balance de: {name} ---")
    
    for i in tqdm(range(len(py_dataset))):
        _, y = py_dataset[i]
        
        fire_in_sample = np.sum(y > 0)
        total_pixels += y.size
        total_fire_pixels += fire_in_sample
        
        if fire_in_sample > 0:
            samples_with_fire += 1

    background_pixels = total_pixels - total_fire_pixels
    fire_ratio = (total_fire_pixels / total_pixels) * 100
    samples_with_fire_pct = (samples_with_fire / len(py_dataset)) * 100

    print(f"\nResultados para {name}:")
    print(f"  - Total de muestras: {len(py_dataset)}")
    print(f"  - Muestras que contienen AL MENOS 1 píxel de fuego: {samples_with_fire} ({samples_with_fire_pct:.2f}%)")
    print(f"  - Píxeles totales analizados: {total_pixels:,}")
    print(f"  - Píxeles de fuego: {total_fire_pixels:,}")
    print(f"  - Píxeles de fondo: {background_pixels:,}")
    print(f"  - RATIO DE FUEGO (Pixel-wise): {fire_ratio:.4f}%")
    
    if fire_ratio < 1.0:
        print("  AVISO: Desbalance crítico. Se recomienda usar pesos en la Loss Function (Pos Weight).")
    
    plt.figure(figsize=(8, 5))
    plt.bar(['Fondo', 'Fuego'], [background_pixels, total_fire_pixels], color=['gray', 'red'])
    plt.yscale('log') 
    plt.title(f"Desbalance de Clases en {name} (Escala Log)")
    plt.ylabel("Número de Píxeles")
    plt.show()

analyze_dataset_balance(train_dataset, name="Train Set")
analyze_dataset_balance(val_dataset, name="Validation Set")